# Patrón de Comportamiento: Mediator

## Introducción
El patrón Mediator define un objeto que encapsula cómo interactúan un conjunto de objetos, promoviendo el bajo acoplamiento.

## Objetivos
- Comprender cómo centralizar la comunicación entre objetos.
- Identificar cuándo es útil el patrón Mediator.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Chat grupal**
En un chat grupal, el mediador (servidor) gestiona la comunicación entre los usuarios, evitando que se comuniquen directamente.

**¿Dónde se usa en proyectos reales?**
En sistemas de chat, controladores de UI, sistemas de tráfico aéreo, etc.

## Sin patrón Mediator (forma errónea)
Los objetos se comunican directamente, generando alto acoplamiento.

In [1]:
class Usuario:
    def __init__(self, nombre):
        self.nombre = nombre
    def enviar(self, mensaje, receptor):
        receptor.recibir(mensaje)
    def recibir(self, mensaje):
        print(f'{self.nombre} recibió: {mensaje}')

# Cada usuario debe conocer a los demás

## Con patrón Mediator (forma correcta)
El mediador centraliza la comunicación entre los objetos.

In [2]:
class MediadorChat:
    def __init__(self):
        self.usuarios = []
    def registrar(self, usuario):
        self.usuarios.append(usuario)
        usuario.mediador = self
    def enviar(self, mensaje, emisor):
        for usuario in self.usuarios:
            if usuario != emisor:
                usuario.recibir(mensaje)

class Usuario:
    def __init__(self, nombre):
        self.nombre = nombre
        self.mediador = None
    def enviar(self, mensaje):
        self.mediador.enviar(mensaje, self)
    def recibir(self, mensaje):
        print(f'{self.nombre} recibió: {mensaje}')

mediador = MediadorChat()
u1 = Usuario('Ana')
u2 = Usuario('Luis')
mediador.registrar(u1)
mediador.registrar(u2)
u1.enviar('Hola a todos')

Luis recibió: Hola a todos


## UML del patrón Mediator
```plantuml
@startuml
class MediadorChat {
    + registrar(usuario)
    + enviar(mensaje, emisor)
}
class Usuario {
    + enviar(mensaje)
    + recibir(mensaje)
}
MediadorChat --> Usuario
Usuario --> MediadorChat
@enduml
```

## Otro ejemplo de la vida real: Formulario de checkout con campos coordinados
**Contexto:** en un checkout de e-commerce, los campos se afectan entre sí: si eliges "recogida en tienda" como método de envío, el pago contraentrega debe deshabilitarse; si aplicas un cupón, el total del pago debe recalcularse. Si cada campo llamara directamente a los demás, terminarías con una telaraña de referencias cruzadas entre todos los campos del formulario (este es, de hecho, el ejemplo clásico con el que el libro original de GoF introduce Mediator: un cuadro de diálogo).

### Sin patrón (forma errónea)
Cada campo debe conocer y llamar directamente a los campos que afecta.

In [3]:
class CampoMetodoEnvio:
    def cambiar(self, valor, campo_pago):
        print(f'Método de envío: {valor}')
        if valor == 'recogida_tienda':
            campo_pago.deshabilitar_contraentrega()

class CampoMetodoPago:
    def deshabilitar_contraentrega(self):
        print('Pago: contraentrega deshabilitado')

# Si mañana agregas un campo "cupón" que también deba avisarle al campo de pago,
# tendrás que ir a CampoMetodoEnvio (y a cada campo relacionado) a cablear la nueva conexión
envio = CampoMetodoEnvio()
pago = CampoMetodoPago()
envio.cambiar('recogida_tienda', pago)

Método de envío: recogida_tienda
Pago: contraentrega deshabilitado


### Con patrón (forma correcta)
Ningún campo conoce a los demás directamente: cada uno solo le avisa al mediador que "algo pasó", y el mediador decide a quién más notificar.

In [4]:
class MediadorCheckout:
    def __init__(self):
        self.campos = {}
    def registrar(self, nombre, campo):
        self.campos[nombre] = campo
        campo.mediador = self
    def notificar(self, emisor, evento):
        if emisor == 'envio' and evento == 'recogida_tienda':
            self.campos['pago'].deshabilitar_contraentrega()
        if emisor == 'cupon' and evento == 'aplicado':
            self.campos['pago'].recalcular_total()

class CampoEnvio:
    def __init__(self):
        self.mediador = None
    def seleccionar(self, opcion):
        print(f'Método de envío: {opcion}')
        if opcion == 'recogida_tienda':
            self.mediador.notificar('envio', 'recogida_tienda')

class CampoPago:
    def __init__(self):
        self.mediador = None
    def deshabilitar_contraentrega(self):
        print('Pago: contraentrega deshabilitado (no aplica en recogida en tienda)')
    def recalcular_total(self):
        print('Pago: recalculando total con cupón aplicado')

class CampoCupon:
    def __init__(self):
        self.mediador = None
    def aplicar(self, codigo):
        print(f'Cupón {codigo} aplicado')
        self.mediador.notificar('cupon', 'aplicado')


mediador = MediadorCheckout()
mediador.registrar('envio', CampoEnvio())
mediador.registrar('pago', CampoPago())
mediador.registrar('cupon', CampoCupon())

mediador.campos['envio'].seleccionar('recogida_tienda')
mediador.campos['cupon'].aplicar('DESCUENTO10')

Método de envío: recogida_tienda
Pago: contraentrega deshabilitado (no aplica en recogida en tienda)
Cupón DESCUENTO10 aplicado
Pago: recalculando total con cupón aplicado


### UML del ejemplo de checkout
```plantuml
@startuml
class MediadorCheckout {
    + registrar(nombre, campo)
    + notificar(emisor, evento)
}
class CampoEnvio {
    + seleccionar(opcion)
}
class CampoPago {
    + deshabilitar_contraentrega()
    + recalcular_total()
}
class CampoCupon {
    + aplicar(codigo)
}
MediadorCheckout --> CampoEnvio
MediadorCheckout --> CampoPago
MediadorCheckout --> CampoCupon
CampoEnvio --> MediadorCheckout
CampoCupon --> MediadorCheckout
@enduml
```

### ¿Dónde más se usa Mediator?
- **Formularios/diálogos de UI:** exactamente este ejemplo — el clásico "dialog box" del libro de GoF, donde un mediador coordina qué controles se habilitan/deshabilitan entre sí.
- **Chat grupal:** el ejemplo con el que abre este notebook — el servidor media entre todos los usuarios conectados.
- **Torres de control de tráfico aéreo:** los aviones no se coordinan entre sí directamente; todos hablan con la torre de control.
- **Orquestadores de microservicios:** un servicio mediador que coordina llamadas entre varios microservicios sin que se conozcan entre sí directamente.
- **Frameworks de UI reactivos:** un "store" central (como Redux) actúa de mediador entre componentes que no se comunican directamente entre sí, sino a través del store.

**Ejercicio de reflexión:** ¿qué pasaría si `CampoPago` también necesitara notificar a `MediadorCheckout` cuando el usuario cambia de tarjeta? ¿Seguiría siendo un diseño de bajo acoplamiento, o el mediador empezaría a saber "demasiado" sobre cada campo?

## Actividad
Crea un mediador para coordinar la comunicación entre diferentes módulos de una aplicación de reservas de vuelos.

---
## Explicación de conceptos clave
- **Bajo acoplamiento:** Los objetos no se comunican directamente.
- **Centralización:** El mediador gestiona la interacción.
- **Aplicación en la vida real:** Útil en chats, controladores de UI y sistemas de tráfico.

## Conclusión
El patrón Mediator es ideal para reducir el acoplamiento y centralizar la comunicación entre objetos.